# ML-04 — Search Intelligence Data Contract

This notebook defines the data contract for the FlyRank content-refresh starter dataset.
Every section states a claim in plain English, then proves it with a query.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
print(f"Loaded: {len(df):,} rows × {len(df.columns)} columns")

Loaded: 30,000 rows × 44 columns


## 1. Unit of analysis + time window

**One row = one pseudonymized content item (page)**, identified by `content_id`.

The dataset contains **30,000 rows across 32 pseudonymized clients**. Each row carries
trailing-90-day aggregate metrics from Google Search Console (GSC) and Google Analytics (GA4),
ending at the export date. The 90-day window is further subdivided into two 30-day comparison
sub-windows (`*_last_30d` = most recent 30 days, `*_prev_30d` = days 31–60 back) used to
compute trend direction.

```
├──── prev 30d ────┤──── last 30d ────┤  (within the trailing 90d window)
     days 60–31          days 30–1        → export date
```

Every content item in this slice has `impressions_90d ≥ 1` and `content_age_days ≥ 90`.

In [2]:
# Grain check: content_id should be unique (zero duplicates = grain holds)
dupes = df.groupby("content_id").size().reset_index(name="c")
dupes = dupes[dupes["c"] > 1]
print(f"Duplicate content_id rows: {len(dupes)}  ← 0 means the grain holds")

# Row count and client count
print(f"Total rows: {len(df):,}")
print(f"Distinct clients: {df['client_id'].nunique()}")

# Every row has impressions ≥ 1 and age ≥ 90
print(f"Min impressions_90d: {df['impressions_90d'].min()}  (expected ≥ 1)")
print(f"Min content_age_days: {df['content_age_days'].min()}  (expected ≥ 90)")

Duplicate content_id rows: 0  ← 0 means the grain holds
Total rows: 30,000
Distinct clients: 32
Min impressions_90d: 1  (expected ≥ 1)
Min content_age_days: 90  (expected ≥ 90)


## 2. Fields: feature / label / context / excluded

Every column is classified into exactly one of four buckets.

### Label / proxy — the target and its sources

| Column | Role | Reason |
|---|---|---|
| `trend_direction` | **Label source** | `is_declining_label = (trend_direction == "down")`. Never a feature. |
| `trend_pct` | **Label source** | Computed from impression comparison windows; `trend_direction` is derived from it. Never a feature. |

### Context — for grouping, joining, splitting; never model inputs

| Column | Role |
|---|---|
| `content_id` | Pseudonymous page ID — unique key, grouping/joins only |
| `client_id` | Pseudonymous client ID — used for client-holdout train/test splits |

### Excluded — never features; each has a reason

| Column | Why excluded |
|---|---|
| `provider_used` | Product-decision flag (which LLM provider generated the content). 71.5% missing. Not a page-performance signal. |
| `model_used` | Product-decision flag (which LLM model name). 19.1% missing. Not a page-performance signal. |
| `impression_tier` | Transparent bucket derived from `impressions_90d` — redundant with the numeric feature itself. |
| `position_tier` | Transparent bucket derived from `avg_position` — redundant with the numeric feature. Also lumps `avg_position=0` ("no data") into `top_3`, which is misleading. |
| `age_tier` | Transparent bucket derived from `content_age_days` — redundant with the numeric feature. |
| `age_tier_order` | Numeric encoding of `age_tier` — same redundancy. |
| `freshness_tier` | Transparent bucket derived from `days_since_last_update` — redundant. |
| `word_count_tier` | Transparent bucket derived from `word_count` — redundant. |
| `char_count_tier` | Transparent bucket derived from `char_count` — redundant. |

### Features — knowable before prediction, safe to use

**Numeric features (used by the pipeline's models):**

| Column | Notes |
|---|---|
| `search_volume` | Keyword search volume estimate. Blank for 2,468 rows (feedly articles have no keyword data). |
| `competition` | Keyword competition score, 0–1. |
| `cpc` | Cost-per-click estimate for the target keyword. |
| `word_count` | Article word count. Blank for 7,699 rows. |
| `char_count` | Article character count. Blank alongside `word_count`. |
| `impressions_90d` → used as `log_impressions_90d` | log1p-transformed (heavy-tailed traffic). |
| `clicks_90d` → used as `log_clicks_90d` | log1p-transformed. |
| `sessions_90d` → used as `log_sessions_90d` | log1p-transformed. |
| `ai_sessions_90d` → used as `log_ai_sessions_90d` | log1p-transformed. |
| `days_with_impressions` | Days in 90-day window with ≥ 1 impression (0–90). |
| `days_with_sessions` | Days in window with ≥ 1 session (0–90). |
| `content_age_days` | Days since content creation. All rows ≥ 90 in this slice. |
| `days_since_last_update` | Days since last content update. |
| `ctr` | clicks / impressions × 100. Rate is a ×100 percentage (0.76 = 0.76%). |
| `avg_position` | Mean GSC position. **`0` means \"no data\", not rank zero** (1,205 rows). |
| `engagement_rate` | engaged_sessions / sessions × 100. |
| `scroll_rate` | scroll_events / pageviews × 100. **Can exceed 100** (119 rows do). |
| `ai_traffic_pct` | ai_sessions / sessions × 100. **Can exceed 100** (23 rows do). |

**Categorical features (used by the pipeline's models):**

| Column | Values |
|---|---|
| `competition_level` | `LOW` / `MEDIUM` / `HIGH` (blank → `unknown` after prep) |
| `content_type` | `keyword article` / `feedly article` / `comparison article` |
| `main_intent` | `informational` / `transactional` / `commercial` / `navigational` (blank → `unknown`) |
| `age_tier` | `31-90`, `91-180`, `181-365`, `365+` (in this slice) |
| `freshness_tier` | `0-30`, `31-90`, `91-180`, `181+` |
| `word_count_tier` | `<1000`, `1000-2000`, `2000-3500`, `3500+` (blank → `unknown`) |
| `impression_tier` | `no_data`, `none`, `low`, `moderate`, `good`, `excellent` |
| `position_tier` | `no_data`, `top_3`, `page_1`, `striking`, `page_3_5`, `deep` |

**Engineered features (added by the prep step, 44 → 52 columns):**

| Column | Meaning |
|---|---|
| `is_declining_label` | **The target.** 1 when `trend_direction == "down"` (16,262 rows = 54.2%), else 0. |
| `log_impressions_90d` | `log1p(impressions_90d)` |
| `log_clicks_90d` | `log1p(clicks_90d)` |
| `log_sessions_90d` | `log1p(sessions_90d)` |
| `log_ai_sessions_90d` | `log1p(ai_sessions_90d)` |
| `has_clicks` | 1 when `clicks_90d > 0` |
| `has_ai_sessions` | 1 when `ai_sessions_90d > 0` |
| `measurable_opportunity` | 1 when `impressions_90d ≥ 100` AND `sessions_90d > 0` |

**30-day sub-window columns** (used by baseline scoring, not directly by models):

| Column | Window |
|---|---|
| `impressions_last_30d`, `clicks_last_30d`, `sessions_last_30d` | Most recent 30 days |
| `impressions_prev_30d`, `clicks_prev_30d`, `sessions_prev_30d` | Days 31–60 back |

In [3]:
# Verify: trend_direction and trend_pct are label sources, never features
# The pipeline's MODEL_NUMERIC_FEATURES and MODEL_CATEGORICAL_FEATURES must exclude them.
import sys
sys.path.insert(0, "../../scripts")
from ml_utils import MODEL_NUMERIC_FEATURES, MODEL_CATEGORICAL_FEATURES

all_model_features = MODEL_NUMERIC_FEATURES + MODEL_CATEGORICAL_FEATURES
assert "trend_direction" not in all_model_features, "LEAKAGE: trend_direction in features!"
assert "trend_pct" not in all_model_features, "LEAKAGE: trend_pct in features!"
assert "content_id" not in all_model_features, "content_id should be context, not feature!"
assert "client_id" not in all_model_features, "client_id should be context, not feature!"
assert "provider_used" not in all_model_features, "provider_used is excluded!"
assert "model_used" not in all_model_features, "model_used is excluded!"
print("✓ Label sources (trend_direction, trend_pct) are NOT in model features")
print("✓ Context columns (content_id, client_id) are NOT in model features")
print("✓ Excluded columns (provider_used, model_used) are NOT in model features")

print(f"\nModel uses {len(MODEL_NUMERIC_FEATURES)} numeric + {len(MODEL_CATEGORICAL_FEATURES)} categorical features")
print(f"Numeric: {MODEL_NUMERIC_FEATURES}")
print(f"Categorical: {MODEL_CATEGORICAL_FEATURES}")

✓ Label sources (trend_direction, trend_pct) are NOT in model features
✓ Context columns (content_id, client_id) are NOT in model features
✓ Excluded columns (provider_used, model_used) are NOT in model features

Model uses 18 numeric + 8 categorical features
Numeric: ['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'log_impressions_90d', 'log_clicks_90d', 'log_sessions_90d', 'log_ai_sessions_90d', 'days_with_impressions', 'days_with_sessions', 'content_age_days', 'days_since_last_update', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct']
Categorical: ['competition_level', 'content_type', 'main_intent', 'age_tier', 'freshness_tier', 'word_count_tier', 'impression_tier', 'position_tier']


## 3. Verify it with queries (grain, counts, missing values, windows)

Every contract claim above gets a verification query here.

### 3a. Grain verification

In [4]:
# Grain: one row per content_id — zero duplicates expected
grain_check = df.groupby("content_id").size().reset_index(name="c")
grain_violations = grain_check[grain_check["c"] > 1]
print(f"Grain violations (content_id duplicates): {len(grain_violations)}")
assert len(grain_violations) == 0, "Grain is broken — content_id is not unique!"
print("✓ Grain holds: one row per content_id")

Grain violations (content_id duplicates): 0
✓ Grain holds: one row per content_id


### 3b. Row counts and distributions

In [5]:
# Total rows, clients, content_type breakdown
print(f"Total rows: {len(df):,}")
print(f"Distinct clients: {df['client_id'].nunique()}")
print(f"\nContent type distribution:")
print(df["content_type"].value_counts().to_string())

# Label distribution (trend_direction)
print(f"\nLabel distribution (trend_direction):")
print(df["trend_direction"].value_counts().to_string())

# Declining rate
declining = (df["trend_direction"].str.lower() == "down").sum()
print(f"\nis_declining_label = 1: {declining:,} rows ({declining/len(df)*100:.1f}%)")

Total rows: 30,000
Distinct clients: 32

Content type distribution:
content_type
keyword article       27207
feedly article         2096
comparison article      697

Label distribution (trend_direction):
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152

is_declining_label = 1: 16,262 rows (54.2%)


### 3c. Missing-value audit — overall and by content_type

Missingness is **systematic, not random**: it follows `content_type` lines.
A blind `fillna(0)` would silently encode content type into features.

In [6]:
# Overall missingness (columns with any nulls)
miss = df.isnull().mean().sort_values(ascending=False)
miss = miss[miss > 0]
print("Columns with missing values (overall):")
for col, pct in miss.items():
    print(f"  {col}: {pct*100:.1f}% ({df[col].isnull().sum():,} rows)")

Columns with missing values (overall):
  provider_used: 71.5% (21,438 rows)
  word_count: 25.7% (7,699 rows)
  char_count: 25.7% (7,699 rows)
  word_count_tier: 25.7% (7,699 rows)
  char_count_tier: 25.7% (7,699 rows)
  model_used: 19.1% (5,733 rows)
  trend_pct: 11.3% (3,388 rows)
  competition_level: 8.7% (2,610 rows)
  search_volume: 8.2% (2,468 rows)
  cpc: 8.2% (2,468 rows)
  competition: 8.2% (2,468 rows)
  main_intent: 7.9% (2,374 rows)
  scroll_rate: 0.4% (125 rows)


In [7]:
# Missingness grouped by content_type — exposes the systematic pattern
check_cols = ["search_volume", "competition", "competition_level", "cpc",
              "main_intent", "word_count", "char_count", "scroll_rate",
              "provider_used", "model_used", "trend_pct"]

for ct in sorted(df["content_type"].dropna().unique()):
    sub = df[df["content_type"] == ct]
    print(f"\ncontent_type = '{ct}' ({len(sub):,} rows):")
    any_missing = False
    for col in check_cols:
        if col in df.columns:
            m = sub[col].isnull().mean()
            if m > 0:
                any_missing = True
                print(f"  {col}: {m*100:.1f}% missing")
    if not any_missing:
        print("  (no missing values)")

print("\n✓ Key finding: 'feedly article' rows have 100% missing keyword data")
print("  (search_volume, competition, competition_level, cpc, main_intent)")
print("  → fillna(0) on these columns would silently leak content_type into features")
print("  → the prep step uses has_*-flags instead for safe imputation")


content_type = 'comparison article' (697 rows):
  scroll_rate: 0.3% missing
  provider_used: 83.4% missing
  trend_pct: 0.3% missing

content_type = 'feedly article' (2,096 rows):
  search_volume: 100.0% missing
  competition: 100.0% missing
  competition_level: 100.0% missing
  cpc: 100.0% missing
  main_intent: 100.0% missing
  scroll_rate: 0.0% missing
  provider_used: 70.0% missing
  model_used: 0.0% missing
  trend_pct: 51.9% missing

content_type = 'keyword article' (27,207 rows):
  search_volume: 1.4% missing
  competition: 1.4% missing
  competition_level: 1.9% missing
  cpc: 1.4% missing
  main_intent: 1.0% missing
  word_count: 28.3% missing
  char_count: 28.3% missing
  scroll_rate: 0.4% missing
  provider_used: 71.3% missing
  model_used: 21.1% missing
  trend_pct: 8.5% missing

✓ Key finding: 'feedly article' rows have 100% missing keyword data
  (search_volume, competition, competition_level, cpc, main_intent)
  → fillna(0) on these columns would silently leak content_ty

### 3d. Gotcha verification — sentinels and out-of-range values

In [8]:
# avg_position == 0 is "no data", not rank zero
no_pos = (df["avg_position"] == 0).sum()
print(f"avg_position == 0 ('no data' sentinel): {no_pos:,} rows")

# scroll_rate can exceed 100 (multiple scroll events per pageview)
sr_over_100 = (pd.to_numeric(df["scroll_rate"], errors="coerce") > 100).sum()
print(f"scroll_rate > 100: {sr_over_100} rows")

# ai_traffic_pct can exceed 100 (independent measurement systems)
at_over_100 = (pd.to_numeric(df["ai_traffic_pct"], errors="coerce") > 100).sum()
print(f"ai_traffic_pct > 100: {at_over_100} rows")

# trend_pct is null exactly when impressions_prev_30d == 0 (division by zero)
prev_zero = df[df["impressions_prev_30d"] == 0]
trend_null_in_prev_zero = prev_zero["trend_pct"].isnull().sum()
print(f"\nimpressions_prev_30d == 0: {len(prev_zero):,} rows")
print(f"trend_pct null in those rows: {trend_null_in_prev_zero:,}")
assert len(prev_zero) == trend_null_in_prev_zero, "trend_pct null pattern mismatch!"
print("✓ trend_pct is null exactly when impressions_prev_30d == 0 (no denominator)")

# Rate columns are ×100 percentages — verify ctr range
print(f"\nctr range: {df['ctr'].min()} – {df['ctr'].max()} (×100 percentage; 0.76 = 0.76%)")

avg_position == 0 ('no data' sentinel): 1,205 rows
scroll_rate > 100: 119 rows
ai_traffic_pct > 100: 23 rows

impressions_prev_30d == 0: 3,388 rows
trend_pct null in those rows: 3,388
✓ trend_pct is null exactly when impressions_prev_30d == 0 (no denominator)

ctr range: 0.0 – 100.0 (×100 percentage; 0.76 = 0.76%)


### 3e. Window verification — numeric ranges confirm the 90-day trailing window

In [9]:
# days_with_impressions and days_with_sessions are bounded by 90
print(f"days_with_impressions: min={df['days_with_impressions'].min()}, max={df['days_with_impressions'].max()} (expected 0–90)")
print(f"days_with_sessions: min={df['days_with_sessions'].min()}, max={df['days_with_sessions'].max()} (expected 0–90)")

# 30-day sub-windows: last_30d + prev_30d should be roughly consistent with 90d totals
# (not exactly equal since there's a third 30-day chunk: days 61-90)
print(f"\nimpressions_last_30d max: {df['impressions_last_30d'].max():,}")
print(f"impressions_prev_30d max: {df['impressions_prev_30d'].max():,}")
print(f"impressions_90d max: {df['impressions_90d'].max():,}")

# content_age_days — all rows ≥ 90 in this slice
print(f"\ncontent_age_days: min={df['content_age_days'].min()}, max={df['content_age_days'].max()}")
assert df["content_age_days"].min() >= 90, "Expected all content_age_days ≥ 90!"
print("✓ All content items are ≥ 90 days old in this slice")

days_with_impressions: min=1, max=88 (expected 0–90)
days_with_sessions: min=1, max=90 (expected 0–90)

impressions_last_30d max: 238,796
impressions_prev_30d max: 218,786
impressions_90d max: 517,715

content_age_days: min=90, max=564
✓ All content items are ≥ 90 days old in this slice


## 4. Data limits

What this data can **never** tell you — important boundaries a reader must know:

### 4a. Observational only — no causal claims

This is a cross-sectional snapshot, not an experiment. We observe associations between
content properties and performance trends. We **cannot** claim that changing a word count or
updating a page *causes* a performance change — that requires a randomized experiment or
a credible quasi-experimental design we do not have here.

### 4b. Single time snapshot — no temporal validation

The starter CSV is a single 90-day snapshot. We have no way to test whether patterns observed
here hold across different time periods (seasonality, algorithm updates, market shifts).
The warehouse release (weeks 3+) offers 17 months of daily data for proper temporal validation.

### 4c. Unbalanced panel in the warehouse release

When moving to the full warehouse (~79M rows), per-client history depth differs wildly.
Some clients have 17 months of data, others only 3. Rows before a client's `ga4_data_start`
have GA4 columns zero-filled (`ga4_data_available = FALSE`) — those zeros are not "no
engagement", they're "not yet measured".

### 4d. Window overlap in the query table

The warehouse's `fact_content_query_90d` table covers a fixed 90-day window that overlaps
the snapshot's final months. If the label is defined on the last 30 days, only `*_prev30`
columns are safe features. Always align feature and label windows before using query-table
columns.

### 4e. Pseudonymized data — no real-world identification

All IDs are pseudonyms. No client names, real URLs, or private search queries appear anywhere.
Results should only be reported in aggregated, pseudonymized form.

### 4f. Survivorship bias

This dataset only contains pages with `impressions_90d ≥ 1` and `content_age_days ≥ 90`.
Pages that never gained any search visibility, or that were deleted before reaching 90 days,
are absent. Any findings apply to *surviving, visible content*, not all content ever created.

In [10]:
# Quantify the survivorship filter
print("=== Survivorship filter check ===")
print(f"All rows have impressions_90d >= 1: {(df['impressions_90d'] >= 1).all()}")
print(f"All rows have content_age_days >= 90: {(df['content_age_days'] >= 90).all()}")

# Show how uneven the client distribution is
print("\n=== Pages per client (top 10 and bottom 5) ===")
client_counts = df["client_id"].value_counts()
print("Top 10:")
print(client_counts.head(10).to_string())
print("\nBottom 5:")
print(client_counts.tail(5).to_string())
print(f"\nClient page counts range: {client_counts.min()} – {client_counts.max()}")
print(f"Median pages per client: {client_counts.median():.0f}")
print("→ Client-holdout split (not random row split) is critical to avoid leaking")
print("  client-level patterns from train into test.")

=== Survivorship filter check ===
All rows have impressions_90d >= 1: True
All rows have content_age_days >= 90: True

=== Pages per client (top 10 and bottom 5) ===
Top 10:
client_id
client_19581e27de    7008
client_6208ef0f77    3681
client_4e07408562    2294
client_3fdba35f04    2267
client_f369cb89fc    1796
client_8527a891e2    1194
client_a88a7902cb    1171
client_d4735e3a26    1106
client_7f2253d7e2    1043
client_f74efabef1    1031

Bottom 5:
client_id
client_02d20bbd7e    38
client_0b918943df    35
client_4fc82b26ae    32
client_8b940be7fb    28
client_1a6562590e     3

Client page counts range: 3 – 7008
Median pages per client: 567
→ Client-holdout split (not random row split) is critical to avoid leaking
  client-level patterns from train into test.


## Output statement

**What the analysis hands to a human reviewer:**
A ranked, reason-coded queue of content items scored for declining-traffic risk (or CTR
opportunity, depending on lane), ordered by priority. The queue is a **decision-support
tool** — it surfaces which pages to review first, not what action to take. All claims are
observational, measured, and directional.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.